In [ ]:
import librosa
import IPython.display as ipd

import numpy as np
import pandas as pd
import matplotlib.pylab as plt

### Audio Input

In [ ]:
audio_path = "Input/rec.wav"

In [ ]:
ipd.Audio(audio_path)

In [ ]:
audio, fs = librosa.load(audio_path, sr=None)
time = np.arange(0, len(audio)) / fs
print(f'shape: {audio.shape}\nsampling rate: {fs}')

In [ ]:
pd.Series(audio).plot(figsize=(10,5), lw=1)

In [ ]:
fig, ax = plt.subplots()
ax.plot(time, audio)
ax.set(xlabel='Time (s)', ylabel='Sound Amplitude')
plt.show()

### Adding Silence Regions

In [ ]:
silenced_audio = audio.copy()
for start, end in [(18, 20), (26, 30)]:
    mask = (time >= start) & (time < end)
    silenced_audio[mask] = 0.0

In [ ]:
(silenced_audio == 0.0).sum()

In [ ]:
fig, ax = plt.subplots()
ax.plot(time, silenced_audio)
ax.set(xlabel='Time (s)', ylabel='Sound Amplitude')
plt.show()

### Denoising
Spectral denoise (spectral subtraction) --> suppressing temporal noise through Median Filtering

In [ ]:
from scipy.signal import stft, istft

def spectral_denoise(x, fs):
    f, t, Zxx = stft(x, fs, nperseg=1024, noverlap=512)
    
    magnitude = np.abs(Zxx)
    phase = np.angle(Zxx)

    # Estimate noise from lowest-energy frames
    noise_profile = np.mean(magnitude[:, :10], axis=1, keepdims=True)

    # Subtract noise (simple spectral subtraction)
    clean_mag = magnitude - noise_profile
    clean_mag = np.maximum(clean_mag, 0)

    # Reconstruct
    Zxx_clean = clean_mag * np.exp(1j * phase)
    _, x_clean = istft(Zxx_clean, fs)

    return x_clean

In [ ]:
def suppress_musical_noise_temporal(x, fs):
    f, t, Zxx = stft(x, fs, nperseg=1024, noverlap=512)

    magnitude = np.abs(Zxx)
    phase = np.angle(Zxx)

    clean_mag = magnitude.copy()

    for i in range(2, magnitude.shape[1] - 2):
        clean_mag[:, i] = np.median(magnitude[:, i-2:i+3], axis=1)  # median filtering

    clean_mag = np.maximum(clean_mag, 1e-6)

    Zxx_clean = clean_mag * np.exp(1j * phase)
    _, x_clean = istft(Zxx_clean, fs)

    return x_clean

In [ ]:
denoised = spectral_denoise(audio, fs)
denoised = suppress_musical_noise_temporal(denoised, fs)

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(silenced_audio, label="Noised and Cut", alpha=0.6)
plt.plot(denoised, label="Less Noise", alpha=0.8)
plt.legend()
plt.show()

### Compression Pipeline Functions

In [ ]:
def compute_stft(signal, fs):
    f, t, Zxx = stft(
        signal,
        fs=fs,
        window='hann',
        nperseg=1024,
        noverlap=512
    )
    return f, t, Zxx

In [ ]:
def quantize(magnitude, step=0.001):
    q = np.round(magnitude / step)
    return q, step

In [ ]:
def rle_encode(arr):
    encoded = []
    count = 1
    
    for i in range(1, len(arr)):
        if arr[i] == arr[i-1]:
            count += 1
        else:
            encoded.append((arr[i-1], count))
            count = 1
    
    encoded.append((arr[-1], count))
    return encoded

In [ ]:
def rle_decode(encoded):
    decoded = []
    for value, count in encoded:
        decoded.extend([value] * count)
    return np.array(decoded)

In [ ]:
def compute_snr(original, reconstructed):
    min_len = min(len(original), len(reconstructed))
    original = original[:min_len]
    reconstructed = reconstructed[:min_len]

    noise = original - reconstructed
    snr = 10 * np.log10(np.sum(original**2) / np.sum(noise**2))
    return snr

### Compression 
STFT --> Splitting into bands --> Quanitisation --> Encoding ---> Decoding --> Dequantise --> Reconstruct (Inverse STFT)

In [ ]:
# STFT
f, t, Zxx = stft(denoised, fs)
magnitude = np.abs(Zxx)
phase = np.angle(Zxx)
magnitude[magnitude < 1e-4] = 0

# dividing them into band frequencies
num_bands = 8
bands = np.array_split(magnitude, num_bands, axis=0)

# Quantize

step = 0.0005
scales = 1 + 0.2 * np.linspace(0, 1, num_bands)
q_mag = np.zeros_like(magnitude)

start = 0
for i, band in enumerate(bands):
    end = start + band.shape[0]
    
    step_i = step * scales[i]
    
    q_mag[start:end, :] = np.round(
        magnitude[start:end, :] / step_i
    )
    
    start = end

# Encode
flat = q_mag.flatten().astype(int)
encoded = rle_encode(flat)

# Decode
decoded_flat = rle_decode(encoded)
decoded_mag = decoded_flat.reshape(magnitude.shape)

# Dequantize
reconstructed_mag = np.zeros_like(magnitude)

start = 0
for i, band in enumerate(bands):
    end = start + band.shape[0]
    
    step_i = step * scales[i]
    
    reconstructed_mag[start:end, :] = (
        decoded_mag[start:end, :] * step_i
    )
    
    start = end
    
# Rebuild
Zxx_reconstructed = reconstructed_mag * np.exp(1j * phase)
_, reconstructed_signal = istft(Zxx_reconstructed, fs)

In [ ]:
snr = compute_snr(denoised, reconstructed_signal)
print("SNR:", snr)

In [ ]:
original_size = magnitude.size
compressed_size = len(encoded)

ratio = original_size / compressed_size
print("Compression Ratio:", ratio)

In [ ]:
from IPython.display import Audio

### Original

In [ ]:
Audio(audio, rate=fs)

### Denoised

In [ ]:
Audio(denoised, rate=fs)

### Compressed

In [ ]:
Audio(reconstructed_signal, rate=fs)

In [ ]:
import soundfile as sf
sf.write('Output/denoised.wav', denoised, fs)
sf.write('Output/compressed.wav', reconstructed_signal, fs)